# ACE-Step song generation (Colab T4, 30-second test)

Text-to-music with vocals using [ACE-Step v1-3.5B](https://github.com/ace-step/ACE-Step).

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Run the cells top to bottom. The first run downloads about 8 GB of weights, which takes a few minutes. After that, each 30-second clip takes well under a minute.

Notes for T4:
- T4 has no fast bfloat16 support, so the model runs in **float16**, the same as the official ACE-Step Colab.
- If a clip comes out silent or as noise (float16 overflow), set `DTYPE = "bfloat16"` in step 3, run it again, then run step 5 again. bfloat16 is slower on a T4, but it matches the precision the model was trained in.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

## 2. Install ACE-Step

If pip ends with a **"Restart session"** prompt (because it pinned `transformers`/`datasets` versions), click it, then carry on from step 3. You don't need to reinstall.

In [ ]:
!pip install -q git+https://github.com/ace-step/ACE-Step.git soundfile

## 2b. Hugging Face login (optional)

The ACE-Step weights are public, so you don't need a token. With one, the 8 GB download is faster and avoids anonymous rate limits.

Add the token as a Colab secret. Click the 🔑 **Secrets** icon in the left sidebar, add a secret named `HF_TOKEN`, and turn on **Notebook access**. Don't paste the token into a cell, because it would be saved in the notebook.

In [ ]:
import os
from huggingface_hub import login
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception as e:  # secret missing or notebook access not granted
    token = None
    print(f"No HF_TOKEN secret found ({type(e).__name__}); downloading anonymously.")
if token:
    os.environ["HF_TOKEN"] = token
    login(token=token, add_to_git_credential=False)
    print("Logged in to Hugging Face.")

## 3. Load the model

`CPU_OFFLOAD` keeps only the model stage that is currently running on the GPU. In float16 everything fits in the T4's 15 GB, so it is off by default. Turn it on if you hit CUDA out-of-memory errors.

In [ ]:
import os, time
import torch
DTYPE = "float16"  #@param ["float16", "bfloat16", "float32"]
CPU_OFFLOAD = False  #@param {type:"boolean"}
CHECKPOINT_DIR = "/content/ace_step_checkpoints"  #@param {type:"string"}

# ACE-Step reads this env var and it overrides the constructor's dtype argument.
os.environ["ACE_PIPELINE_DTYPE"] = DTYPE

import numpy as np
import soundfile as sf
from acestep.pipeline_ace_step import ACEStepPipeline

# Newer torchaudio routes save() through torchcodec, which Colab may not have.
# Write the WAV with soundfile directly instead.
def _save_wav_file(self, target_wav, idx, save_path=None, sample_rate=48000, format="wav"):
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    wav = target_wav.float().cpu().numpy().T  # (channels, samples) -> (samples, channels)
    sf.write(save_path, wav, sample_rate)
    return save_path
ACEStepPipeline.save_wav_file = _save_wav_file

t0 = time.time()
pipe = ACEStepPipeline(
    checkpoint_dir=CHECKPOINT_DIR,
    cpu_offload=CPU_OFFLOAD,
    torch_compile=False,
    overlapped_decode=False,
)
pipe.load_checkpoint(pipe.checkpoint_dir)
pipe.loaded = True
print(f"Loaded in {time.time() - t0:.0f}s | dtype={pipe.dtype} | "
      f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 4. Describe the song

- **Prompt:** comma-separated tags for genre, instruments, mood, tempo and vocal style.
- **Lyrics:** use `[verse]`, `[chorus]` and `[bridge]` section tags. For an instrumental, set `LYRICS = "[instrumental]"`.
- 30 seconds covers about one verse and one chorus, so keep the lyrics short.

In [ ]:
PROMPT = "pop, female vocals, acoustic guitar, piano, upbeat, catchy, 110 bpm, bright, warm"  #@param {type:"string"}
DURATION = 30  #@param {type:"slider", min:10, max:60, step:5}
INFER_STEPS = 60  #@param {type:"slider", min:20, max:100, step:5}
GUIDANCE_SCALE = 15.0  #@param {type:"number"}
SEED = 42  #@param {type:"integer"}

LYRICS = """[verse]
Morning light across the floor
Coffee steam and an open door
Every street is calling out my name
Nothing here will ever feel the same

[chorus]
Oh we're running with the summer sun
Hearts on fire, we're just getting started
Oh we're running till the day is done
"""

## 5. Generate

In [ ]:
from IPython.display import Audio, display

os.makedirs("/content/outputs", exist_ok=True)
out_path = f"/content/outputs/acestep_{time.strftime('%Y%m%d_%H%M%S')}_seed{SEED}.wav"

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
result = pipe(
    format="wav",
    audio_duration=float(DURATION),
    prompt=PROMPT,
    lyrics=LYRICS,
    infer_step=INFER_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    scheduler_type="euler",
    cfg_type="apg",
    omega_scale=10.0,
    manual_seeds=[SEED],
    guidance_interval=0.5,
    guidance_interval_decay=0.0,
    min_guidance_scale=3.0,
    use_erg_tag=True,
    use_erg_lyric=True,
    use_erg_diffusion=True,
    oss_steps=None,
    guidance_scale_text=0.0,
    guidance_scale_lyric=0.0,
    save_path=out_path,
    batch_size=1,
)
elapsed = time.time() - t0
wav_path = result[0]

# Sanity check: float16 overflow shows up as NaN/Inf, silence, or full-scale noise.
audio, sr = sf.read(wav_path)
finite = bool(np.isfinite(audio).all())
peak = float(np.nanmax(np.abs(audio))) if audio.size else 0.0
rms = float(np.sqrt(np.nanmean(audio ** 2))) if audio.size else 0.0
print(f"Generated {len(audio) / sr:.1f}s in {elapsed:.0f}s -> {wav_path}")
print(f"peak={peak:.3f}  rms={rms:.4f}  peak VRAM={torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
if not finite or peak < 1e-3 or rms > 0.5:
    print("WARNING: output looks broken (NaN, silence or noise). "
          "Set DTYPE='bfloat16' in step 3, run it again, then run this cell again.")

display(Audio(wav_path))

## 6. Download (optional)

In [ ]:
from google.colab import files
files.download(wav_path)

## 7. Save to Google Drive (optional)

This can also cache the model weights on Drive so the next session skips the 8 GB download. To use the cache, set `CHECKPOINT_DIR = "/content/drive/MyDrive/ace_step_checkpoints"` in step 3.

In [ ]:
from google.colab import drive
import shutil
CACHE_WEIGHTS = False  #@param {type:"boolean"}

drive.mount("/content/drive")
dst = "/content/drive/MyDrive/ace_step_outputs"
os.makedirs(dst, exist_ok=True)
shutil.copy(wav_path, dst)
print("Copied to", dst)

if CACHE_WEIGHTS and not CHECKPOINT_DIR.startswith("/content/drive"):
    shutil.copytree(CHECKPOINT_DIR, "/content/drive/MyDrive/ace_step_checkpoints", dirs_exist_ok=True)
    print("Weights cached to /content/drive/MyDrive/ace_step_checkpoints")